In [2]:
import polars as pl

In [3]:
! pwd

/home/phinguyen/de_upskills_mock_prj


In [4]:
! ls data/MovieLens

genome_scores.csv  genome_tags.csv  link.csv  movie.csv  rating.csv  tag.csv


# Movie

In [5]:
movie = pl.read_csv("data/MovieLens/movie.csv")

In [6]:
movie

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""
…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy"""
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy"""
131258,"""The Pirates (2014)""","""Adventure"""


In [7]:
# Check unique values in the "genres" column
movie.select(
    pl.col("genres").unique().alias("unique_genres")
)

unique_genres
str
"""Adventure|Drama|Mystery"""
"""Horror|Sci-Fi|Thriller"""
"""Adventure|Musical"""
"""Action|Comedy|Fantasy|IMAX"""
"""Action|Documentary|Drama|Roman…"
…
"""Action|Adventure|Fantasy"""
"""Action|Crime|Film-Noir"""
"""Children|Comedy|Drama|Musical"""


In [8]:
# sample of genres is "Comedy|Horror|Mystery|Sci-Fi"
# transform the genres column to rows, for example, "Comedy|Horror|Mystery|Sci-Fi" will be transformed to 4 rows: "Comedy", "Horror", "Mystery", "Sci-Fi"

movie_exploded = movie.with_columns(
    pl.col("genres").str.split("|").alias("genres_list")
).explode("genres_list").rename({"genres_list": "genre"})

movie_exploded

/tmp/ipykernel_6047/3391097171.py:6: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  ).explode("genres_list").rename({"genres_list": "genre"})


movieId,title,genres,genre
i64,str,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Adventure"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Animation"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Children"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Comedy"""
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Fantasy"""
…,…,…,…
131258,"""The Pirates (2014)""","""Adventure""","""Adventure"""
131260,"""Rentun Ruusu (2001)""","""(no genres listed)""","""(no genres listed)"""
131262,"""Innocence (2014)""","""Adventure|Fantasy|Horror""","""Adventure"""


In [9]:
movie_exploded.select(
    pl.col("genre").unique().alias("unique_genre")
).write_csv("data/investigation/movie_unique_genres.csv")

In [10]:
movie = movie.with_columns(
    pl.col("title")
    .str.extract(r"^(.*) \((\d{4})\)$", group_index=1)
    .alias("movie_name"),
    pl.col("title")
    .str.extract(r"^(.*) \((\d{4})\)$", group_index=2)
    .cast(pl.Int32)
    .alias("year"),
)

movie

movieId,title,genres,movie_name,year
i64,str,str,str,i32
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Toy Story""",1995
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""","""Jumanji""",1995
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""","""Grumpier Old Men""",1995
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""","""Waiting to Exhale""",1995
5,"""Father of the Bride Part II (1…","""Comedy""","""Father of the Bride Part II""",1995
…,…,…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy""","""Kein Bund für's Leben""",2007
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy""","""Feuer, Eis & Dosenbier""",2002
131258,"""The Pirates (2014)""","""Adventure""","""The Pirates""",2014


In [11]:
year_pattern = r"^(.*) \((\d{4})\)$"

unmatched_titles = (
    movie
    .with_columns(
        pl.col("title")
        .str.extract(year_pattern, group_index=1)
        .alias("movie_name"),
        pl.col("title")
        .str.extract(year_pattern, group_index=2)
        .cast(pl.Int32)
        .alias("year"),
    )
    .filter(pl.col("year").is_null())
    .select("title", "movie_name", "year")
)

unmatched_titles.write_csv("data/investigation/unmatched_titles.csv")

In [12]:
year_pattern = r"^(.*?)[\s\u00A0]*\((\d{4})\)$"

parsed = movie.with_columns(
    pl.col("title")
    .str.strip_chars()
    .str.extract(year_pattern, group_index=1)
    .alias("movie_name"),
    pl.col("title")
    .str.strip_chars()
    .str.extract(year_pattern, group_index=2)
    .cast(pl.Int32)
    .alias("year"),
)

unmatched_titles = parsed.filter(pl.col("year").is_null())

In [13]:
unmatched_titles.write_csv("data/investigation/unmatched_titles2.csv")

In [14]:
import polars as pl

title = pl.col("title").str.strip_chars()

# Accept a terminal year with normal spacing, or one extra closing parenthesis.
year_text = title.str.extract(r"\((\d{4})\)\)?$", group_index=1)

# TV-style ranges are flagged separately; don't treat their start year as movie release year.
has_year_range = title.str.contains(r"\(\d{4}\s*[-–]\s*\d{4}?\s*\)$")

result = movie.with_columns(
    title.str.replace(r"\s*\(\d{4}\)\)?$", "").alias("movie_name"),
    year_text.cast(pl.Int32).alias("year"),
    has_year_range.fill_null(False).alias("has_year_range"),
).with_columns(
    pl.when(pl.col("year").is_not_null())
      .then(pl.lit("parsed"))
      .when(pl.col("has_year_range"))
      .then(pl.lit("year_range_review"))
      .otherwise(pl.lit("year_missing"))
      .alias("year_parse_status")
)

In [15]:
result

movieId,title,genres,movie_name,year,has_year_range,year_parse_status
i64,str,str,str,i32,bool,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…","""Toy Story""",1995,false,"""parsed"""
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy""","""Jumanji""",1995,false,"""parsed"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance""","""Grumpier Old Men""",1995,false,"""parsed"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance""","""Waiting to Exhale""",1995,false,"""parsed"""
5,"""Father of the Bride Part II (1…","""Comedy""","""Father of the Bride Part II""",1995,false,"""parsed"""
…,…,…,…,…,…,…
131254,"""Kein Bund für's Leben (2007)""","""Comedy""","""Kein Bund für's Leben""",2007,false,"""parsed"""
131256,"""Feuer, Eis & Dosenbier (2002)""","""Comedy""","""Feuer, Eis & Dosenbier""",2002,false,"""parsed"""
131258,"""The Pirates (2014)""","""Adventure""","""The Pirates""",2014,false,"""parsed"""


# Source schema and CDM profiling

This section gathers the evidence needed before defining the lakehouse contracts. The assignment asks you to profile what is actually present; its listed column types and relationships are expectations to validate, not values to assume.

For each file, capture: (1) source columns and inferred physical types; (2) row count, null/blank counts, ranges and representative values; (3) candidate key uniqueness and grain; (4) domain and relationship checks; and (5) meaning, timestamp semantics, and whether a field is source-entered or derived. These findings inform a raw-preserving Bronze schema, a typed/conformed Silver CDM, and Gold facts/dimensions. Avoid changing raw source data during profiling.

## 1. Actual schema, nulls, value ranges, and examples

The following lazy profiler scans each source and writes two reviewable outputs under `data/investigation/`. It does not calculate exact cardinality for every column (which would be costly for 20M-row identifiers); key uniqueness is checked separately below.

In [16]:
from pathlib import Path
import polars as pl

DATA_DIR = Path("data/MovieLens")
OUT_DIR = Path("data/investigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    "rating": "rating.csv",
    "tag": "tag.csv",
    "movie": "movie.csv",
    "link": "link.csv",
    "genome_scores": "genome_scores.csv",
    "genome_tags": "genome_tags.csv",
}

def source_scan(file_name: str) -> pl.LazyFrame:
    # Keep the source timestamp representation as-is for this physical schema profile.
    return pl.scan_csv(DATA_DIR / file_name, infer_schema_length=10_000, try_parse_dates=False, null_values=[""])

source_frames = {name: source_scan(file_name) for name, file_name in SOURCE_FILES.items()}

schema_rows = []
column_rows = []
for table_name, lf in source_frames.items():
    print(f"Scanning {table_name}...")
    schema = lf.collect_schema()
    # One aggregate per source: total rows and null counts in a single scan.
    null_exprs = [pl.col(c).null_count().alias(f"null__{c}") for c in schema.names()]
    summary = lf.select([pl.len().alias("row_count"), *null_exprs]).collect().to_dicts()[0]
    for column_name, dtype in schema.items():
        schema_rows.append({
            "table": table_name, "source_file": SOURCE_FILES[table_name],
            "column": column_name, "polars_inferred_type": str(dtype),
            "row_count": summary["row_count"],
            "null_count": summary[f"null__{column_name}"],
        })
        expr = pl.col(column_name)
        # Min/max aid numeric/date range checks; for strings they are lexical, so treat as hints only.
        stats = lf.select(
            expr.min().cast(pl.String).alias("min_value"),
            expr.max().cast(pl.String).alias("max_value"),
            expr.drop_nulls().head(5).cast(pl.String).alias("sample_values"),
        ).collect().to_dicts()[0]
        column_rows.append({
            "table": table_name, "column": column_name,
            "min_value": stats["min_value"], "max_value": stats["max_value"],
            "sample_values": stats["sample_values"],
        })

schema_profile = pl.DataFrame(schema_rows).sort("table", "column")
column_profile = pl.DataFrame(column_rows).sort("table", "column")
schema_profile.write_csv(OUT_DIR / "schema_profile.csv")
column_profile.write_csv(OUT_DIR / "column_value_profile.csv")
schema_profile, column_profile


Scanning rating...
Scanning tag...
Scanning movie...
Scanning link...
Scanning genome_scores...
Scanning genome_tags...


(shape: (19, 6)
 ┌───────────────┬───────────────────┬───────────┬──────────────────────┬───────────┬────────────┐
 │ table         ┆ source_file       ┆ column    ┆ polars_inferred_type ┆ row_count ┆ null_count │
 │ ---           ┆ ---               ┆ ---       ┆ ---                  ┆ ---       ┆ ---        │
 │ str           ┆ str               ┆ str       ┆ str                  ┆ i64       ┆ i64        │
 ╞═══════════════╪═══════════════════╪═══════════╪══════════════════════╪═══════════╪════════════╡
 │ genome_scores ┆ genome_scores.csv ┆ movieId   ┆ Int64                ┆ 11709768  ┆ 0          │
 │ genome_scores ┆ genome_scores.csv ┆ relevance ┆ Float64              ┆ 11709768  ┆ 0          │
 │ genome_scores ┆ genome_scores.csv ┆ tagId     ┆ Int64                ┆ 11709768  ┆ 0          │
 │ genome_tags   ┆ genome_tags.csv   ┆ tag       ┆ String               ┆ 1128      ┆ 0          │
 │ genome_tags   ┆ genome_tags.csv   ┆ tagId     ┆ Int64                ┆ 1128      ┆ 0      

## 2. Confirm candidate keys and table grain

A grain is what one row represents. Check the expected source keys empirically. Ratings and tags are event tables: repeated user/movie combinations can be legitimate, so exact row duplication and candidate-key behavior are separate checks. Genome scores should have one row per `(movieId, tagId)`.

In [17]:
# Key profile: reports row count, distinct key count, and duplicate excess.
# Exact distinct counts on these small composite key sets are intentional full-data checks.
KEYS = {
    "rating": ["userId", "movieId"],
    "tag": ["userId", "movieId", "tag", "timestamp"],
    "movie": ["movieId"],
    "link": ["movieId"],
    "genome_scores": ["movieId", "tagId"],
    "genome_tags": ["tagId"],
}

key_rows = []
for table_name, key_columns in KEYS.items():
    lf = source_frames[table_name]
    key_stats = lf.select(
        pl.len().alias("row_count"),
        pl.struct(key_columns).n_unique().alias("distinct_key_count"),
        pl.struct(key_columns).is_duplicated().sum().alias("rows_marked_duplicate"),
    ).collect().to_dicts()[0]
    key_rows.append({
        "table": table_name, "candidate_key": ", ".join(key_columns),
        **key_stats,
        "row_count_minus_distinct_keys": key_stats["row_count"] - key_stats["distinct_key_count"],
    })
key_profile = pl.DataFrame(key_rows)
key_profile.write_csv(OUT_DIR / "key_profile.csv")
key_profile


table,candidate_key,row_count,distinct_key_count,rows_marked_duplicate,row_count_minus_distinct_keys
str,str,i64,i64,i64,i64
"""rating""","""userId, movieId""",20000263,20000263,0,0
"""tag""","""userId, movieId, tag, timestam…",465564,465564,0,0
"""movie""","""movieId""",27278,27278,0,0
"""link""","""movieId""",27278,27278,0,0
"""genome_scores""","""movieId, tagId""",11709768,11709768,0,0
"""genome_tags""","""tagId""",1128,1128,0,0


## 3. Type/domain checks and relationship evidence

Use the results to choose explicit Silver types and DQ rules. In particular, validate timestamp parsing and event ranges, rating increments/range, genome relevance range, IMDb/TMDb nullability, and foreign keys. Raw strings remain available in Bronze even when parsing fails.

In [18]:
# Parse event timestamps as a derived profiling expression. Source CSV columns remain unchanged.
def with_event_time(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns(
        pl.col("timestamp").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("event_time")
    )

rating_checks = with_event_time(source_frames["rating"]).select(
    pl.len().alias("rows"),
    pl.col("rating").null_count().alias("rating_nulls"),
    ((pl.col("rating") < 0.5) | (pl.col("rating") > 5.0) | ((pl.col("rating") * 2) % 1 != 0)).fill_null(False).sum().alias("invalid_rating_rows"),
    pl.col("event_time").null_count().alias("timestamp_parse_failures"),
    pl.col("event_time").min().alias("event_time_min"),
    pl.col("event_time").max().alias("event_time_max"),
).collect()

tag_checks = with_event_time(source_frames["tag"]).select(
    pl.len().alias("rows"),
    pl.col("tag").null_count().alias("null_tags"),
    (pl.col("tag").str.strip_chars().fill_null("") == "").sum().alias("empty_or_whitespace_tags"),
    pl.col("event_time").null_count().alias("timestamp_parse_failures"),
).collect()

genome_score_checks = source_frames["genome_scores"].select(
    pl.len().alias("rows"),
    pl.col("relevance").null_count().alias("null_relevance"),
    ((pl.col("relevance") < 0) | (pl.col("relevance") > 1)).fill_null(False).sum().alias("out_of_range_relevance"),
).collect()

movie_checks = source_frames["movie"].select(
    pl.len().alias("rows"),
    pl.col("title").null_count().alias("null_titles"),
    pl.col("genres").null_count().alias("null_genres"),
    (pl.col("genres") == "(no genres listed)").sum().alias("no_genres_sentinel_rows"),
).collect()

link_checks = source_frames["link"].select(
    pl.len().alias("rows"),
    pl.col("imdbId").null_count().alias("null_imdb_ids"),
    pl.col("tmdbId").null_count().alias("null_tmdb_ids"),
).collect()
rating_checks, tag_checks, movie_checks, genome_score_checks, link_checks


(shape: (1, 6)
 ┌──────────┬──────────────┬──────────────────┬─────────────────┬─────────────────┬─────────────────┐
 │ rows     ┆ rating_nulls ┆ invalid_rating_r ┆ timestamp_parse ┆ event_time_min  ┆ event_time_max  │
 │ ---      ┆ ---          ┆ ows              ┆ _failures       ┆ ---             ┆ ---             │
 │ u32      ┆ u32          ┆ ---              ┆ ---             ┆ datetime[μs]    ┆ datetime[μs]    │
 │          ┆              ┆ u32              ┆ u32             ┆                 ┆                 │
 ╞══════════╪══════════════╪══════════════════╪═════════════════╪═════════════════╪═════════════════╡
 │ 20000263 ┆ 0            ┆ 0                ┆ 0               ┆ 1995-01-09      ┆ 2015-03-31      │
 │          ┆              ┆                  ┆                 ┆ 11:46:44        ┆ 06:40:02        │
 └──────────┴──────────────┴──────────────────┴─────────────────┴─────────────────┴─────────────────┘,
 shape: (1, 4)
 ┌────────┬───────────┬──────────────────────────┬─

In [19]:
# Referential integrity anti-joins. These can take time because the large files are scanned.
def anti_join_count(child: pl.LazyFrame, child_key: str, parent: pl.LazyFrame, parent_key: str) -> int:
    return child.join(
        parent.select(pl.col(parent_key).alias("_parent_key")).unique(),
        left_on=child_key, right_on="_parent_key", how="anti",
    ).select(pl.len()).collect().item()

relationship_profile = pl.DataFrame({
    "relationship": [
        "rating.movieId -> movie.movieId", "tag.movieId -> movie.movieId",
        "link.movieId -> movie.movieId", "movie.movieId -> link.movieId",
        "genome_scores.movieId -> movie.movieId", "genome_scores.tagId -> genome_tags.tagId",
    ],
    "orphan_rows": [
        anti_join_count(source_frames["rating"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["tag"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["link"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["movie"], "movieId", source_frames["link"], "movieId"),
        anti_join_count(source_frames["genome_scores"], "movieId", source_frames["movie"], "movieId"),
        anti_join_count(source_frames["genome_scores"], "tagId", source_frames["genome_tags"], "tagId"),
    ],
})
relationship_profile.write_csv(OUT_DIR / "relationship_profile.csv")
relationship_profile


relationship,orphan_rows
str,i64
"""rating.movieId -> movie.movieI…",0
"""tag.movieId -> movie.movieId""",0
"""link.movieId -> movie.movieId""",0
"""movie.movieId -> link.movieId""",0
"""genome_scores.movieId -> movie…",0
"""genome_scores.tagId -> genome_…",0


## 4. Turn profiles into model decisions

Use measured output rather than assumptions to complete a mapping document. A starting model hypothesis (to validate, not blindly adopt):

- **Rating event:** one row per rating interaction; preserve `rating` on the source 0.5–5 scale, record normalized event time in UTC, and keep source system/key lineage. Do not assume `(userId, movieId)` is unique until the key profile confirms the intended semantics.
- **User tag event:** one row per tag submission; preserve raw text and create a separately normalized tag value in Silver. The key may need timestamp or a source row identifier because users can repeat tags.
- **Movie:** conformed content entity keyed by MovieLens `movieId`; split title, extracted release year, genre bridge rows, and external IDs into suitable Silver/Gold structures. Retain the source title and parsing status.
- **Genome score:** derived movie–genome-tag measurement, grain `(movieId, tagId)`; keep distinct from user-submitted tags.
- **Link:** movie identifier crosswalk; test the actual one-to-one coverage and permit missing external IDs where observed.
- **Bronze:** append-only, source-shaped typed data plus batch/file/hash/ingestion audit fields; do not cleanse or drop malformed values here.
- **Silver:** explicit types, normalized/conformed names and units, deterministic deduplication, DQ quarantine, and incremental MERGE. Add derived columns without overwriting raw source values.
- **Gold:** declare fact grain and use surrogate-key dimensions. Movie genre history needs Type 2 if point-in-time genre analysis is required; decide Type 1/3 per attribute based on analytical need.

Save your actual schema, key, relationship, and domain outputs alongside a short explanation of any deviations from the assignment's expected counts/types.

# Follow-up investigations for design decisions

The cells below measure the open items called out in `docs/data_model_design.md`. Run each section independently if you want to manage scan time. Results are written to `data/investigation/`.

## A. Timestamp parseability and timezone evidence

The CSV timestamps contain no explicit timezone offset in the documented format. This code measures parse failures and detects whether any values include timezone markers. Even if every value parses, the data alone cannot establish whether the naive clock is UTC or another timezone; confirm that from source documentation or the dataset provider before naming a Silver field `event_time_utc`.

In [34]:
from pathlib import Path
import polars as pl

OUT_DIR = Path("data/investigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def timestamp_profile(table_name: str) -> pl.DataFrame:
    lf = source_frames[table_name]
    parsed = pl.col("timestamp").str.to_datetime(
        format="%Y-%m-%d %H:%M:%S", strict=False
    )
    return lf.select(
        pl.len().alias("rows"),
        pl.col("timestamp").null_count().alias("source_nulls"),
        parsed.null_count().alias("parse_failures_including_nulls"),
        parsed.min().alias("parsed_min"),
        parsed.max().alias("parsed_max"),
        pl.col("timestamp").str.contains(r"(?i)(Z|[+-]\d{2}:?\d{2})$").fill_null(False).sum().alias("rows_with_explicit_timezone_marker"),
        pl.col("timestamp").str.contains(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$").fill_null(False).sum().alias("rows_matching_naive_timestamp_shape"),
    ).with_columns(pl.lit(table_name).alias("table")).collect()

timestamp_results = pl.concat([timestamp_profile("rating"), timestamp_profile("tag")]).select(
    "table", "rows", "source_nulls", "parse_failures_including_nulls",
    "parsed_min", "parsed_max", "rows_with_explicit_timezone_marker",
    "rows_matching_naive_timestamp_shape",
)
timestamp_results.write_csv(OUT_DIR / "timestamp_profile.csv")
timestamp_results


table,rows,source_nulls,parse_failures_including_nulls,parsed_min,parsed_max,rows_with_explicit_timezone_marker,rows_matching_naive_timestamp_shape
str,u32,u32,u32,datetime[μs],datetime[μs],u32,u32
"""rating""",20000263,0,0,1995-01-09 11:46:44,2015-03-31 06:40:02,0,20000263
"""tag""",465564,0,0,2005-12-24 13:00:10,2015-03-31 03:09:12,0,465564


## B. Rating domain, distribution, and skew

This checks every rating for the assignment's [0.5, 5.0] range and half-star increments, reports frequency by rating value, and summarizes ratings per user/movie. Skew summaries help choose join strategy and avoid high-cardinality partition keys.

In [35]:
rating_lf = source_frames["rating"]
rating_domain = rating_lf.select(
    pl.len().alias("rows"),
    ((pl.col("rating") < 0.5) | (pl.col("rating") > 5.0)).fill_null(False).sum().alias("outside_0_5_to_5"),
    (((pl.col("rating") * 2) - (pl.col("rating") * 2).round(0)).abs() > 1e-8).fill_null(False).sum().alias("not_half_star_increment"),
    pl.col("rating").null_count().alias("null_ratings"),
).collect()
rating_distribution = rating_lf.group_by("rating").agg(pl.len().alias("rows")).sort("rating").collect()

def skew_summary(group_key: str, metric: str) -> pl.DataFrame:
    per_entity = rating_lf.group_by(group_key).agg(pl.len().alias("event_count"))
    return per_entity.select(
        pl.lit(metric).alias("metric"), pl.len().alias("entities"),
        pl.col("event_count").min().alias("min"),
        pl.col("event_count").quantile(0.50).alias("p50"),
        pl.col("event_count").quantile(0.95).alias("p95"),
        pl.col("event_count").quantile(0.99).alias("p99"),
        pl.col("event_count").max().alias("max"),
        pl.col("event_count").mean().alias("mean"),
    ).collect()

rating_skew = pl.concat([
    skew_summary("movieId", "ratings_per_movie"),
    skew_summary("userId", "ratings_per_user"),
])
rating_domain.write_csv(OUT_DIR / "rating_domain_profile.csv")
rating_distribution.write_csv(OUT_DIR / "rating_distribution.csv")
rating_skew.write_csv(OUT_DIR / "rating_skew_profile.csv")
rating_domain, rating_distribution, rating_skew


(shape: (1, 4)
 ┌──────────┬──────────────────┬─────────────────────────┬──────────────┐
 │ rows     ┆ outside_0_5_to_5 ┆ not_half_star_increment ┆ null_ratings │
 │ ---      ┆ ---              ┆ ---                     ┆ ---          │
 │ u32      ┆ u32              ┆ u32                     ┆ u32          │
 ╞══════════╪══════════════════╪═════════════════════════╪══════════════╡
 │ 20000263 ┆ 0                ┆ 0                       ┆ 0            │
 └──────────┴──────────────────┴─────────────────────────┴──────────────┘,
 shape: (10, 2)
 ┌────────┬─────────┐
 │ rating ┆ rows    │
 │ ---    ┆ ---     │
 │ f64    ┆ u32     │
 ╞════════╪═════════╡
 │ 0.5    ┆ 239125  │
 │ 1.0    ┆ 680732  │
 │ 1.5    ┆ 279252  │
 │ 2.0    ┆ 1430997 │
 │ 2.5    ┆ 883398  │
 │ 3.0    ┆ 4291193 │
 │ 3.5    ┆ 2200156 │
 │ 4.0    ┆ 5561926 │
 │ 4.5    ┆ 1534824 │
 │ 5.0    ┆ 2898660 │
 └────────┴─────────┘,
 shape: (2, 8)
 ┌───────────────────┬──────────┬─────┬──────┬────────┬─────────┬───────┬─────────

## C. Exact duplicate rows in each static source file

Candidate-key uniqueness does not detect a duplicate row under a different key definition. This computes full-row distinct counts for each source. It is a full scan and can be costly for large CSVs. Repeated rows across *different landing batches* are a separate incremental-ingestion check and should be tested against the batch simulator.

In [36]:
duplicate_rows = []
for table_name, lf in source_frames.items():
    columns = lf.collect_schema().names()
    stats = lf.select(
        pl.len().alias("rows"),
        pl.struct(columns).n_unique().alias("distinct_full_rows"),
    ).collect().to_dicts()[0]
    duplicate_rows.append({
        "table": table_name,
        **stats,
        "exact_duplicate_excess_rows": stats["rows"] - stats["distinct_full_rows"],
    })
exact_duplicate_profile = pl.DataFrame(duplicate_rows).sort("table")
exact_duplicate_profile.write_csv(OUT_DIR / "exact_duplicate_profile.csv")
exact_duplicate_profile


table,rows,distinct_full_rows,exact_duplicate_excess_rows
str,i64,i64,i64
"""genome_scores""",11709768,11709768,0
"""genome_tags""",1128,1128,0
"""link""",27278,27278,0
"""movie""",27278,27278,0
"""rating""",20000263,20000263,0
"""tag""",465564,465564,0


## D. User-tag normalization impact

Compare raw tag values with a conservative canonical form: Unicode NFC normalization, trim, lowercase, and collapse repeated whitespace. This intentionally does not strip punctuation or merge synonyms such as `sci-fi` and `scifi`; those require a governed mapping. Review blank values and normalized groups that combine multiple raw spellings before adopting the rule.

In [37]:
tag_lf = source_frames["tag"]
normalized_tag = (
    pl.col("tag")
    .str.normalize(form="NFC")
    .str.strip_chars()
    .str.to_lowercase()
    .str.replace_all(r"\s+", " ")
)
tag_normalized = tag_lf.with_columns(normalized_tag.alias("tag_normalized"))
tag_norm_summary = tag_normalized.select(
    pl.len().alias("rows"),
    pl.col("tag").n_unique().alias("distinct_raw_tag_values"),
    pl.col("tag_normalized").n_unique().alias("distinct_normalized_tag_values"),
    (pl.col("tag").str.strip_chars().fill_null("") == "").sum().alias("blank_or_whitespace_rows"),
    (pl.col("tag").fill_null("") != pl.col("tag_normalized").fill_null("")).sum().alias("rows_changed_by_normalization"),
).collect()

# Groups with multiple raw spellings after conservative normalization are candidates for review.
normalization_collisions = (
    tag_normalized.group_by("tag_normalized")
    .agg(
        pl.col("tag").n_unique().alias("distinct_raw_variants"),
        pl.len().alias("rows"),
    )
    .filter(pl.col("distinct_raw_variants") > 1)
    .sort("distinct_raw_variants", descending=True)
    .collect()
)
# Export collision examples in long form so every CSV column is a scalar.
normalization_collision_examples = (
    tag_normalized.select("tag_normalized", "tag")
    .unique()
    .join(
        normalization_collisions.lazy().select("tag_normalized"),
        on="tag_normalized",
        how="inner",
    )
    .sort("tag_normalized", "tag")
    .collect()
)
tag_norm_summary.write_csv(OUT_DIR / "tag_normalization_summary.csv")
normalization_collisions.write_csv(OUT_DIR / "tag_normalization_collisions.csv")
normalization_collision_examples.write_csv(OUT_DIR / "tag_normalization_collision_examples.csv")
tag_norm_summary, normalization_collisions.head(30), normalization_collision_examples.head(100)


(shape: (1, 5)
 ┌────────┬──────────────────────┬──────────────────────┬─────────────────────┬─────────────────────┐
 │ rows   ┆ distinct_raw_tag_val ┆ distinct_normalized_ ┆ blank_or_whitespace ┆ rows_changed_by_nor │
 │ ---    ┆ ues                  ┆ tag_values           ┆ _rows               ┆ malization          │
 │ u32    ┆ ---                  ┆ ---                  ┆ ---                 ┆ ---                 │
 │        ┆ u32                  ┆ u32                  ┆ u32                 ┆ u32                 │
 ╞════════╪══════════════════════╪══════════════════════╪═════════════════════╪═════════════════════╡
 │ 465564 ┆ 38644                ┆ 35162                ┆ 7                   ┆ 162625              │
 └────────┴──────────────────────┴──────────────────────┴─────────────────────┴─────────────────────┘,
 shape: (30, 3)
 ┌────────────────────┬───────────────────────┬──────┐
 │ tag_normalized     ┆ distinct_raw_variants ┆ rows │
 │ ---                ┆ ---               

## E. Genre coverage and title parse status

The genre profile counts valid labels and the no-genre sentinel separately, and surfaces labels outside the documented set. Title statuses distinguish a clean terminal year, a year range, an extra closing parenthesis after a terminal year, and no recognizable terminal year. These statuses are investigation flags; do not invent missing release years.

In [38]:
movie_lf = source_frames["movie"]
allowed_genres = [
    "Action", "Adventure", "Animation", "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western", "IMAX",
]

genre_rows = (
    movie_lf.filter(pl.col("genres") != "(no genres listed)")
    .select(pl.col("genres").str.split("|").alias("genre_list"))
    .explode("genre_list")
    .with_columns(pl.col("genre_list").str.strip_chars().alias("genre"))
    .group_by("genre")
    .agg(pl.len().alias("movie_genre_associations"))
    .sort("movie_genre_associations", descending=True)
    .collect()
)
genre_quality = movie_lf.select(
    pl.len().alias("movie_rows"),
    (pl.col("genres") == "(no genres listed)").sum().alias("no_genres_sentinel_rows"),
    pl.col("genres").null_count().alias("null_genres"),
).collect()
unknown_genres = genre_rows.filter(~pl.col("genre").is_in(allowed_genres))

title = pl.col("title").str.strip_chars()
year = title.str.extract(r"\((\d{4})\)\)?$", group_index=1)
is_range = title.str.contains(r"\(\d{4}\s*[-–]\s*\d{0,4}\s*\)$").fill_null(False)
has_year_like_suffix = title.str.contains(r"\([^)]*\d{4}[^)]*\)\)?$").fill_null(False)
title_profile = movie_lf.with_columns(
    year.alias("extracted_year_text"),
    is_range.alias("is_year_range"),
    has_year_like_suffix.alias("has_year_like_suffix"),
).with_columns(
    pl.when(pl.col("extracted_year_text").is_not_null()).then(pl.lit("parsed_terminal_year"))
    .when(pl.col("is_year_range")).then(pl.lit("year_range_review"))
    .when(pl.col("has_year_like_suffix")).then(pl.lit("malformed_year_suffix_review"))
    .otherwise(pl.lit("no_recognizable_terminal_year")).alias("title_parse_status")
)
title_status_counts = title_profile.group_by("title_parse_status").agg(pl.len().alias("movies")).sort("title_parse_status").collect()
title_review_examples = title_profile.filter(pl.col("title_parse_status") != "parsed_terminal_year").select(
    "movieId", "title", "title_parse_status"
).collect()

genre_rows.write_csv(OUT_DIR / "genre_frequency_profile.csv")
genre_quality.write_csv(OUT_DIR / "genre_quality_profile.csv")
unknown_genres.write_csv(OUT_DIR / "unknown_genre_profile.csv")
title_status_counts.write_csv(OUT_DIR / "title_parse_status_profile.csv")
title_review_examples.write_csv(OUT_DIR / "title_parse_review_rows.csv")
genre_quality, genre_rows, unknown_genres, title_status_counts, title_review_examples.head(30)


/tmp/ipykernel_6047/1894590523.py:11: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("genre_list")


(shape: (1, 3)
 ┌────────────┬─────────────────────────┬─────────────┐
 │ movie_rows ┆ no_genres_sentinel_rows ┆ null_genres │
 │ ---        ┆ ---                     ┆ ---         │
 │ u32        ┆ u32                     ┆ u32         │
 ╞════════════╪═════════════════════════╪═════════════╡
 │ 27278      ┆ 246                     ┆ 0           │
 └────────────┴─────────────────────────┴─────────────┘,
 shape: (19, 2)
 ┌───────────┬──────────────────────────┐
 │ genre     ┆ movie_genre_associations │
 │ ---       ┆ ---                      │
 │ str       ┆ u32                      │
 ╞═══════════╪══════════════════════════╡
 │ Drama     ┆ 13344                    │
 │ Comedy    ┆ 8374                     │
 │ Thriller  ┆ 4178                     │
 │ Romance   ┆ 4127                     │
 │ Action    ┆ 3520                     │
 │ …         ┆ …                        │
 │ Musical   ┆ 1036                     │
 │ Animation ┆ 1027                     │
 │ Western   ┆ 676            

## F. Genome score range, coverage, and approximate file sizes

The score check verifies the full [0,1] range. Coverage reports how many catalog movies have at least one genome score and how many scores each covered movie has. File sizes are source CSV sizes on disk; converted Parquet/Iceberg sizes will differ and should be measured after writing a representative sample.

In [39]:
genome_lf = source_frames["genome_scores"]
movie_ids_lf = movie_lf.select("movieId").unique()
genome_domain = genome_lf.select(
    pl.len().alias("score_rows"),
    pl.col("relevance").min().alias("min_relevance"),
    pl.col("relevance").max().alias("max_relevance"),
    pl.col("relevance").null_count().alias("null_relevance"),
    ((pl.col("relevance") < 0) | (pl.col("relevance") > 1)).fill_null(False).sum().alias("out_of_range_rows"),
).collect()
genome_coverage_counts = genome_lf.select(pl.col("movieId").n_unique().alias("movies_with_genome")).collect()
catalog_count = movie_ids_lf.select(pl.len().alias("catalog_movies")).collect()
genome_coverage = genome_coverage_counts.join(catalog_count, how="cross").with_columns(
    (pl.col("movies_with_genome") / pl.col("catalog_movies") * 100).alias("catalog_coverage_percent")
)
scores_per_movie = genome_lf.group_by("movieId").agg(pl.len().alias("score_count")).select(
    pl.len().alias("covered_movies"),
    pl.col("score_count").min().alias("min_scores_per_movie"),
    pl.col("score_count").quantile(0.50).alias("p50_scores_per_movie"),
    pl.col("score_count").quantile(0.99).alias("p99_scores_per_movie"),
    pl.col("score_count").max().alias("max_scores_per_movie"),
).collect()

file_size_profile = pl.DataFrame([
    {"table": table_name, "source_file": file_name, "bytes": (DATA_DIR / file_name).stat().st_size,
     "megabytes_decimal": (DATA_DIR / file_name).stat().st_size / 1_000_000}
    for table_name, file_name in SOURCE_FILES.items()
]).sort("bytes", descending=True)
genome_domain.write_csv(OUT_DIR / "genome_domain_profile.csv")
genome_coverage.write_csv(OUT_DIR / "genome_coverage_profile.csv")
scores_per_movie.write_csv(OUT_DIR / "genome_scores_per_movie_profile.csv")
file_size_profile.write_csv(OUT_DIR / "source_file_size_profile.csv")
genome_domain, genome_coverage, scores_per_movie, file_size_profile


(shape: (1, 5)
 ┌────────────┬───────────────┬───────────────┬────────────────┬───────────────────┐
 │ score_rows ┆ min_relevance ┆ max_relevance ┆ null_relevance ┆ out_of_range_rows │
 │ ---        ┆ ---           ┆ ---           ┆ ---            ┆ ---               │
 │ u32        ┆ f64           ┆ f64           ┆ u32            ┆ u32               │
 ╞════════════╪═══════════════╪═══════════════╪════════════════╪═══════════════════╡
 │ 11709768   ┆ 0.00025       ┆ 1.0           ┆ 0              ┆ 0                 │
 └────────────┴───────────────┴───────────────┴────────────────┴───────────────────┘,
 shape: (1, 3)
 ┌────────────────────┬────────────────┬──────────────────────────┐
 │ movies_with_genome ┆ catalog_movies ┆ catalog_coverage_percent │
 │ ---                ┆ ---            ┆ ---                      │
 │ u32                ┆ u32            ┆ f64                      │
 ╞════════════════════╪════════════════╪══════════════════════════╡
 │ 10381              ┆ 27278     

## Interpretation checklist

After running the cells, update the design document with measured outputs. Explicitly state that timezone meaning is a source-level assumption unless the provider confirms it. Choose `event_time_utc` only after establishing source timezone; until then use a neutral parsed timestamp name such as `event_time_source`. Use tag normalization counts/collision examples to justify a conservative Silver rule. Use rating skew quantiles and actual query patterns to justify partitioning; never partition by every user or movie ID. Keep exact duplicate checks on simulated reruns separate from duplicates in the original snapshot.

## Choose the simulator cutoff and event window

Review monthly volume before selecting `T`. Prefer a cutoff on a month boundary that leaves enough post-cutoff data for several batches; compare both rating and tag volumes. The source timestamps have no timezone marker, so this uses their source wall-clock month consistently and does not convert to UTC.

In [ ]:
monthly_volume = pl.concat([
    source_frames["rating"].select(
        pl.lit("rating").alias("source"),
        pl.col("timestamp").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).dt.strftime("%Y-%m").alias("event_month"),
    ),
    source_frames["tag"].select(
        pl.lit("tag").alias("source"),
        pl.col("timestamp").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).dt.strftime("%Y-%m").alias("event_month"),
    ),
]).group_by("source", "event_month").agg(pl.len().alias("rows")).sort("source", "event_month").with_columns(
    pl.col("rows").cum_sum().over("source").alias("cumulative_rows")
).collect()
monthly_volume.write_csv(OUT_DIR / "monthly_event_volume.csv")
monthly_volume
